# Lectio - Transcripción de clases

**Javier Otero**

**UCAB, Escuela de Ingeniería de Informática**

**Herramienta de apoyo al estudio · v0.4**

Este notebook transcribe las grabaciones de una semana de clases y archiva cada resultado
en la carpeta de su materia. Corre sobre la GPU que Colab asigna de forma gratuita.

El motor es [`faster-whisper`](https://github.com/SYSTRAN/faster-whisper), una
reimplementación de Whisper sobre CTranslate2. El modelo es el mismo que publica OpenAI;
lo que cambia es el motor de inferencia, que aprovecha mejor el hardware.

## El problema que resuelve

Una clase de dos horas transcrita en una CPU de escritorio toma del orden de nueve horas,
lo que vuelve impracticable el flujo diario. La misma transcripción sobre una GPU T4 baja
a minutos, y esa diferencia es la que justifica mover el procesamiento a Colab.

## La estructura de trabajo

```
MyDrive/google_colab/transcriptioner/
├── audios/
│   ├── week_1/
│   ├── week_2/
│   │   └── ucab-w2-d4-apispy-24-09-2026.m4a
│   └── week_3/
│       └── ucab-w3-d4-apispy-01-10-2026.m4a
├── text/
│   ├── apispy/
│   │   ├── ucab-w2-d4-apispy-24-09-2026.md
│   │   └── ucab-w3-d4-apispy-01-10-2026.md
│   ├── bd/
│   └── ciberoff/
└── oido_transcription.ipynb
```

**La entrada se organiza por semana y la salida por materia**, y cada criterio responde a
un uso distinto.

Los audios entran por semana porque así es como se graban y se suben: al terminar la
semana hay un lote que procesar, y la carpeta acota exactamente ese lote. Preguntarse
"¿qué me falta transcribir?" es preguntarse por una semana.

Las transcripciones salen por materia porque así es como se estudian. Al preparar un
parcial de bases de datos no interesa qué semana fue cada clase: interesan todas las de
bases de datos juntas, en orden. Una carpeta por materia es también la unidad que se
puede entregar a un asistente para que responda sobre esa materia, y eso es lo que
desarrolla el Anexo C.

El nombre de archivo es lo que conecta las dos vistas: lleva la semana y la materia, de
modo que cada transcripción sabe de qué lote vino y a qué carpeta va.

## Cómo está organizado

| Sección | Qué hace |
|---|---|
| 1 | Verifica que Colab haya asignado una GPU |
| 2 | Instala el motor y resuelve las librerías de NVIDIA |
| 3 | Conecta Drive y selecciona la semana a procesar |
| 4 | Interpreta los nombres de archivo y valida |
| 5 | Transcribe y archiva por materia |
| 6 | Resume el estado por semana y por materia |

Las secciones 1 y 2 se ejecutan una vez por sesión. De la 3 en adelante se repiten por
cada semana que se quiera procesar: basta cambiar el número y volver a ejecutar.

## 1. Entorno de ejecución

Colab asigna el hardware al crear la sesión, no al ejecutar las celdas. Si la sesión se
creó sin acelerador, esta celda lo detecta y el resto del notebook no tiene sentido:
`faster-whisper` funciona igual sobre CPU, pero a la velocidad que se intenta evitar.

### Decisiones de implementación

* **La verificación va primero y es bloqueante.** Descubrir la falta de GPU después de
  haber descargado tres gigabytes de pesos es una pérdida de tiempo evitable.
* **Se consulta `nvidia-smi` y no `torch.cuda`.** El motor no usa PyTorch, de modo que
  preguntarle a PyTorch por el estado de la GPU introduciría una dependencia que el
  notebook no necesita para nada más.
* **Se reporta la memoria disponible**, porque determina qué modelo entra. Una T4 tiene
  15 GB y admite `large-v3` sin problema; aceleradores más chicos obligan a bajar de
  modelo.
* **Se contemplan las dos formas de no tener GPU.** En un entorno sin acelerador
  `nvidia-smi` no está instalado, de modo que la llamada no devuelve un código de error:
  lanza `FileNotFoundError`. Mirar solo el código de salida dejaría la celda reventando
  con un traceback en vez de dar el diagnóstico.

In [ ]:
# ============================================
# 1. Entorno de ejecución
# ============================================

import subprocess

# --------------------------------------------
# Procesamiento
# --------------------------------------------

# En un entorno sin acelerador, `nvidia-smi` no existe como programa: la llamada no
# devuelve un código de error, lanza FileNotFoundError. Hay que contemplar las dos
# formas de "no hay GPU".
try:
    consulta = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
        capture_output=True, text=True,
    )
    hay_gpu = consulta.returncode == 0
except FileNotFoundError:
    consulta, hay_gpu = None, False

# --------------------------------------------
# Logs
# --------------------------------------------

if hay_gpu:
    nombre, memoria, driver = [campo.strip() for campo in consulta.stdout.strip().split(',')]
    print("Acelerador    : ", nombre)
    print("Memoria       : ", memoria)
    print("Driver NVIDIA : ", driver)
    print("\nEntorno listo. Continuá con la sección 2.")
else:
    print("NO HAY GPU EN ESTA SESIÓN.")
    print()
    print("Esta celda es bloqueante a propósito: sin acelerador, una clase de dos horas")
    print("tarda horas en vez de minutos, y la sección 2 va a fallar al construir el modelo.")
    print()
    print("Dos causas posibles:")
    print()
    print("  1. La sesión se creó sin acelerador.")
    print("     Entorno de ejecución -> Cambiar tipo de entorno -> GPU, y volvé a ejecutar")
    print("     esta celda. El cambio reinicia la sesión, pero todavía no hay nada que perder.")
    print()
    print("  2. Ya elegiste GPU y aun así no hay.")
    print("     Es el límite de uso de la cuenta gratuita, que se agota tras corridas largas")
    print("     y se repone con el tiempo. No se puede forzar: hay que esperar, o correr")
    print("     mientras tanto con otra cuenta.")
    print()
    print("Lo ya transcrito está en Drive y no se pierde. Cuando vuelva la GPU, se retoma")
    print("desde la sección 3 y se omite todo lo que ya tenga su archivo escrito.")

### Qué deberías ver

El nombre del acelerador asignado, normalmente `Tesla T4` con 15360 MiB. Cualquier GPU de
esa familia o superior sirve.

Si aparece el aviso de que no hay GPU, cambiá el tipo de entorno y volvé a ejecutar **esta
misma celda**: el cambio reinicia la sesión, pero no hay nada que perder todavía porque
aún no se instaló ni se descargó nada.

## 2. Instalación del motor

Se instala `faster-whisper` junto con las librerías de NVIDIA que necesita, y se resuelve
un problema de enlazado que es específico de Colab.

### El problema de cuDNN

`ctranslate2`, el motor sobre el que corre `faster-whisper`, requiere CUDA 12.3 o superior
con cuDNN 9. Las versiones que Colab trae preinstaladas no siempre coinciden, y cuando no
coinciden el error aparece recién al construir el modelo, con un mensaje que no sugiere
ninguna solución:

```
Unable to load any of {libcudnn_cnn.so.9.1.0, libcudnn_cnn.so.9.1, libcudnn_cnn.so.9}
Invalid handle. Cannot load symbol cudnnCreateConvolutionDescriptor
```

La solución habitual que circula en los foros es exportar `LD_LIBRARY_PATH` apuntando al
directorio de las librerías. **Esa solución no funciona desde un notebook**, y la razón es
que el enlazador dinámico de glibc lee esa variable una sola vez, cuando el proceso
arranca. Para cuando una celda la modifica, el intérprete de Python ya está corriendo y el
valor nuevo no se vuelve a consultar.

### Decisiones de implementación

* **Se cargan las librerías con `ctypes.CDLL` y `RTLD_GLOBAL`** en lugar de manipular la
  variable de entorno. Esto las trae al espacio de direcciones del proceso en ejecución y
  las publica en el ámbito global, que es donde `ctranslate2` las va a buscar cuando se
  cargue a sí mismo. Es la misma operación que haría el enlazador, hecha a mano y a
  tiempo.
* **Se recorre el árbol de `dist-packages/nvidia` con un patrón**, porque la ruta exacta
  cambia entre versiones de las ruedas de NVIDIA y fijarla a mano rompería el notebook en
  cuanto Colab actualice una dependencia.
* **Los fallos de carga individuales se ignoran.** El árbol contiene librerías para
  arquitecturas que esta GPU no usa; que algunas no carguen es lo esperado, no un error.
* **La celda termina construyendo un modelo `tiny` sobre CUDA.** Es la única verificación
  que vale: si ese modelo se construye, el enlazado quedó resuelto. Ocupa unos 75 MB y se
  descarta enseguida.
* **PyAV queda fijado por debajo de la versión 19.** PyAV es la librería que decodifica
  el audio, y su versión 19 eliminó el parámetro `metadata_errors` de `av.open()`, que
  `faster-whisper` todavía le pasa. Sin la restricción, la instalación funciona, el modelo
  se construye, y recién al transcribir el primer archivo aparece
  `TypeError: open() got an unexpected keyword argument 'metadata_errors'` — con todos los
  archivos fallando uno tras otro. Es un fallo tardío y desconcertante, por eso conviene
  prevenirlo en la instalación.

In [ ]:
# ============================================
# 2. Instalación del motor
# ============================================

# `av<19` no es caprichoso: PyAV 19 eliminó el parámetro `metadata_errors` de
# `av.open()`, y faster-whisper todavía lo pasa al decodificar el audio. Sin la
# restricción, cada archivo falla con:
#   TypeError: open() got an unexpected keyword argument 'metadata_errors'
!pip install -q faster-whisper 'av<19' nvidia-cublas-cu12 nvidia-cudnn-cu12

import ctypes
import glob
import os

# --------------------------------------------
# Configuración
# --------------------------------------------

# Familias de librerías que ctranslate2 necesita resueltas antes de cargarse
PATRONES_LIBRERIAS = ('libcublas*.so*', 'libcublasLt*.so*', 'libcudnn*.so*')

RAIZ_NVIDIA = '/usr/local/lib/python3*/dist-packages/nvidia'

# --------------------------------------------
# Procesamiento
# --------------------------------------------

# Se publican las librerías en el ámbito global del proceso. RTLD_GLOBAL es lo que hace
# que los símbolos queden visibles para ctranslate2 cuando se cargue después.
cargadas, omitidas = [], 0

for patron in PATRONES_LIBRERIAS:
    for ruta in glob.glob(f'{RAIZ_NVIDIA}/**/{patron}', recursive=True):
        try:
            ctypes.CDLL(ruta, mode=ctypes.RTLD_GLOBAL)
            cargadas.append(os.path.basename(ruta))
        except OSError:
            # Librerías para arquitecturas que esta GPU no usa. Se esperan.
            omitidas += 1

# Verificación real del enlazado: si este modelo se construye, la GPU quedó operativa.
from faster_whisper import WhisperModel

_sonda = WhisperModel('tiny', device='cuda', compute_type='float16')
del _sonda

# --------------------------------------------
# Logs
# --------------------------------------------

print("Librerías publicadas :", len(cargadas))
print("Librerías omitidas   :", omitidas, "(normal: arquitecturas que esta GPU no usa)")
print()
print("Verificación con un modelo de prueba sobre CUDA: correcta.")
print("El motor quedó operativo.")

### Qué deberías ver

La cuenta de librerías publicadas (una docena, aproximadamente) y la confirmación de que
el modelo de prueba se construyó.

Van a aparecer también dos advertencias de Hugging Face sobre peticiones sin autenticar.
**Son informativas y no afectan en nada**: avisan que una cuenta daría límites de descarga
más altos, lo cual es irrelevante para bajar un modelo una vez por sesión.

Si la celda falla en la construcción del modelo, reiniciá el entorno con
`Entorno de ejecución → Reiniciar sesión` y ejecutala de nuevo. El reinicio limpia el
estado del enlazador, que es lo que suele quedar inconsistente cuando Colab trae librerías
preinstaladas en conflicto.

## 3. Drive y selección de las semanas

Se conecta Drive y se eligen las semanas a procesar. **Es el único parámetro que cambia
entre una corrida y la siguiente.**

### El alcance de la corrida

`SEMANAS` admite tres formas:

| Valor | Qué procesa |
|---|---|
| `[]` | Todas las semanas que tengan audios |
| `[1]` | Solo la semana 1 |
| `[1, 3, 4]` | Esas tres |

Dejarlo vacío es lo habitual cuando hay atraso acumulado: recorre todo lo que haya y
omite lo que ya tenga su transcripción escrita. Acotar a una lista sirve para rehacer una
semana puntual sin volver a recorrer el resto.

### ¿Por qué las semanas se indican a mano?

Porque es el dato que acota la corrida, y conviene que sea una decisión explícita. Se sabe
de antemano qué carpetas se van a tocar antes de ejecutar nada.

Una alternativa sería deducirlas de la fecha de cada grabación, pero eso obligaría a
declarar cuándo empezó el semestre y a mantener esa referencia correcta todo el período.
**La semana ya está escrita en el nombre del archivo y en la carpeta que lo contiene**, de
modo que deducirla sería recalcular algo que ya está dicho dos veces.

### ¿Por qué todo se escribe en Drive y no en el disco de Colab?

El sistema de archivos de Colab vive en la máquina virtual asignada a la sesión y
desaparece con ella. La sesión termina cuando se cierra la pestaña, cuando el navegador
pierde la conexión o cuando Colab recicla el entorno por inactividad, y ninguna de las
tres cosas avisa. Una transcripción de ocho clases escrita en `/content` se pierde entera
si la sesión cae a mitad de la madrugada.

Drive sobrevive a todo eso. El costo es que las escrituras son más lentas, pero se escribe
un archivo de texto por clase: la diferencia es despreciable frente al riesgo.

### Decisiones de implementación

* **Las carpetas de semana se leen de Drive, no se asumen.** Pedir una semana que no
  existe produce un aviso con la lista de las que sí, en lugar de un error.
* **No hay fecha de inicio de semestre.** La semana y el día vienen del nombre del
  archivo; la fecha queda como dato de referencia para el lector.
* **Las carpetas con nombre irregular se reportan aparte.** Drive crea duplicados
  llamados `week_7 (1)` al subir dos veces la misma carpeta; interpretarlas como `week_7`
  haría que sus audios se contaran dos veces.

In [ ]:
# ============================================
# 3. Drive y selección de las semanas
# ============================================

import re
from pathlib import Path

from google.colab import drive

# --------------------------------------------
# Configuración
# --------------------------------------------

# Semanas a procesar.  []  = todas las que tengan audios
#                      [1] = solo la 1
#                   [1, 3] = esas dos
SEMANAS = []

# Raíz de la estructura dentro de Drive
RAIZ = Path('/content/drive/MyDrive/google_colab/transcriptioner')

DIR_AUDIOS = RAIZ / 'audios'
DIR_TEXTO  = RAIZ / 'text'

# --------------------------------------------
# Procesamiento
# --------------------------------------------

drive.mount('/content/drive')


def numero_de_semana(carpeta):
    """Número de una carpeta 'week_N', o None si el nombre no es exactamente eso.

    Drive crea duplicados llamados 'week_7 (1)' cuando se sube dos veces la misma
    carpeta. Esos nombres no se interpretan como semanas: se reportan aparte para que
    se limpien, en lugar de contarse dos veces o romper el ordenamiento.
    """
    coincide = re.fullmatch(r'week_(\d+)', carpeta.name)
    return int(coincide.group(1)) if coincide else None


def semanas_ordenadas(directorio):
    """Las carpetas week_N válidas, ordenadas, y aparte las de nombre irregular."""
    if not directorio.is_dir():
        return [], []
    validas, irregulares = [], []
    for carpeta in directorio.iterdir():
        if not carpeta.is_dir() or not carpeta.name.startswith('week'):
            continue
        (validas if numero_de_semana(carpeta) is not None else irregulares).append(carpeta)
    return sorted(validas, key=numero_de_semana), sorted(irregulares)


DIR_TEXTO.mkdir(parents=True, exist_ok=True)

todas, irregulares = semanas_ordenadas(DIR_AUDIOS)

if SEMANAS:
    seleccionadas = [c for c in todas if numero_de_semana(c) in SEMANAS]
    inexistentes  = sorted(set(SEMANAS) - {numero_de_semana(c) for c in todas})
else:
    seleccionadas = todas
    inexistentes  = []

# Las materias conocidas son las carpetas que ya existen en text/. Sirven para detectar
# un nombre mal escrito antes de que cree una carpeta nueva por error.
materias_conocidas = sorted(p.name for p in DIR_TEXTO.iterdir() if p.is_dir())

# --------------------------------------------
# Logs
# --------------------------------------------

print("Raíz    :", RAIZ)
print("Salida  :", DIR_TEXTO, "/ <materia>")
print()

print("Materias con carpeta en text/:")
if materias_conocidas:
    for materia in materias_conocidas:
        n = len(list((DIR_TEXTO / materia).glob('*.md')))
        print(f"  {materia:<16} {n:>2} transcripción(es)")
else:
    print("  ninguna todavía. Se van a crear a medida que aparezcan.")

print()
print("Semanas en audios/:")
for carpeta in todas:
    n = len(list(carpeta.glob('*')))
    marca = '  <- seleccionada' if carpeta in seleccionadas else ''
    print(f"  {carpeta.name:<14} {n:>2} archivo(s){marca}")

if not todas:
    print("  ninguna. Revisá que la estructura exista en Drive.")

for carpeta in irregulares:
    print(f"  {carpeta.name:<14} (nombre irregular: se ignora)")

if inexistentes:
    print()
    print(f"  AVISO: pediste {inexistentes} pero esas carpetas no existen en audios/.")

print()
alcance = f"semanas {SEMANAS}" if SEMANAS else "todas las semanas"
print(f"Alcance de la corrida: {alcance} -> "
      f"{len(seleccionadas)} carpeta(s) a recorrer")
if seleccionadas:
    print("Continuá con la sección 4.")

### Qué deberías ver

Las rutas confirmadas y el listado de materias que ya tienen carpeta, con cuántas
transcripciones lleva cada una.

Ese listado cumple una segunda función además de informar: la sección 4 lo usa para
detectar materias mal escritas. Si una grabación trae una materia que no está en esa
lista, lo avisa antes de crear una carpeta nueva.

Si aparece que no existe la carpeta de la semana, el listado de semanas disponibles te
dice qué hay realmente en Drive.

## 4. Convención de nombres y validación

Los nombres de archivo no son decorativos: son los metadatos de cada grabación, y de ellos
sale la carpeta de destino. Esta sección los interpreta y verifica antes de transcribir.

### La convención

```
ucab-w2-d4-apispy-24-09-2026.m4a
│    │  │  │      └─ fecha de la clase, DD-MM-YYYY
│    │  │  └──────── materia: decide la carpeta de destino en text/
│    │  └─────────── día de la semana, 1 = lunes … 7 = domingo
│    └────────────── semana: debe coincidir con la carpeta en audios/
└─────────────────── institución
```

### Qué se verifica y qué no

La fecha **no se usa para decidir nada**. Está en el nombre como referencia para quien lo
lea: permite saber de qué día fue una clase sin abrir el archivo. Las divisiones las hacen
`w#` y `d#`, que se toman tal cual vienen.

Eso no impide verificar la coherencia entre ambos, y conviene hacerlo porque no cuesta
nada:

* **`d#` contra la fecha.** El día de la semana de una fecha es intrínseco a esa fecha: el
  24-09-2026 es jueves y no hace falta ninguna referencia externa para saberlo. Si el
  nombre dice `d4` y la fecha cae en martes, uno de los dos está mal escrito.
* **`w#` contra la carpeta.** Un archivo guardado en `audios/week_2/` cuyo nombre dice
  `w5` está mal archivado, mal nombrado, o ambas.
* **La materia contra las carpetas de `text/`.** Si una grabación trae una materia que
  nunca apareció antes, lo más probable es que sea un error de tipeo —`apispyy` por
  `apispy`— y sin aviso crearía una carpeta nueva que parece legítima.

Ninguna de las tres necesita saber cuándo empezó el semestre.

### Decisiones de implementación

* **Las validaciones advierten, no bloquean.** Un nombre incoherente se transcribe igual y
  el aviso queda en la salida. Negarse a procesar obligaría a volver a subir el archivo
  por un guion mal puesto.
* **La materia no admite guiones, y eso vuelve exacto el patrón.** Un nombre con un
  campo de más —`ucab-w2-d4-apispy-lab-24-09-2026`— simplemente no coincide, y va a
  `_sin_materia` con su aviso. Si la materia admitiera guiones, ese mismo nombre se
  interpretaría como una materia llamada `apispy-lab` y crearía una carpeta que aparenta
  ser legítima.
* **La materia determina la carpeta, tal cual está escrita.** No se normaliza ni se
  corrige por parecido: adivinar que dos nombres similares son el mismo lleva a fusionar
  materias que sí eran distintas. Lo que sí se hace es avisar, que es la verificación de
  más abajo.
* **Se admite el formato anterior** (`ucab-apispy-24-09-2026`, sin `w#` ni `d#`). Se
  parsea igual y se archiva por su materia; la semana se toma de la carpeta.
* **Lo que no parsea va a `_sin_materia`.** Se transcribe igual, porque perder una clase
  por un nombre mal escrito sería peor que tener un archivo en una carpeta provisional.
* **Las semanas se recorren en orden** y, dentro de cada una, los archivos por nombre. El
  orden importa poco para el resultado pero mucho para seguir una corrida larga: se sabe
  en todo momento por dónde va.
* **Lo ya transcrito se omite.** El criterio es la existencia del archivo de salida en la
  carpeta de su materia, lo que mantiene al notebook sin estado: la verdad está en Drive.

In [ ]:
# ============================================
# 4. Convención de nombres y validación
# ============================================

from datetime import date

# --------------------------------------------
# Configuración
# --------------------------------------------

EXTENSIONES_AUDIO = {'.mp3', '.wav', '.m4a', '.ogg', '.opus', '.flac', '.aac', '.webm'}

FORMATO_SALIDA = 'md'   # md | txt | srt | vtt

# Carpeta para las grabaciones cuyo nombre no permite determinar la materia
SIN_MATERIA = '_sin_materia'

# Verificaciones de coherencia. Desactivar con False para silenciar los avisos.
VALIDAR = True

# La materia no lleva guiones: el guion es el separador de campos. Eso hace que el
# patrón sea exacto y que un nombre mal formado no coincida, en vez de absorber los
# campos de más dentro de la materia.
PATRON_NUEVO = re.compile(
    r'^(?P<institucion>[^-]+)-w(?P<semana>\d{1,2})-d(?P<dia>[1-7])-(?P<materia>[^-]+)'
    r'-(?P<d>\d{2})-(?P<m>\d{2})-(?P<a>\d{4})$'
)

# Formato anterior, sin semana ni día.
PATRON_VIEJO = re.compile(
    r'^(?P<institucion>[^-]+)-(?P<materia>[^-]+)-(?P<d>\d{2})-(?P<m>\d{2})-(?P<a>\d{4})$'
)

DIAS_SEMANA = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo']

# --------------------------------------------
# Procesamiento
# --------------------------------------------

def interpretar(nombre_sin_extension, semana_carpeta):
    """Descompone el nombre en sus metadatos.

    `semana_carpeta` es el número de la carpeta que contiene el archivo. Se usa como
    valor por omisión para el formato viejo, que no lleva la semana en el nombre.
    """
    for patron, etiqueta in ((PATRON_NUEVO, 'nuevo'), (PATRON_VIEJO, 'viejo')):
        encontrado = patron.match(nombre_sin_extension)
        if encontrado is None:
            continue

        grupos = encontrado.groupdict()
        try:
            fecha = date(int(grupos['a']), int(grupos['m']), int(grupos['d']))
        except ValueError:
            # Tiene forma de fecha pero no corresponde a ninguna del calendario.
            continue

        return {
            'formato'        : etiqueta,
            'institucion'    : grupos['institucion'],
            'materia'        : grupos['materia'],
            'fecha'          : fecha,
            'semana'         : int(grupos['semana']) if 'semana' in grupos else semana_carpeta,
            'semana_carpeta' : semana_carpeta,
            'dia'            : int(grupos['dia']) if 'dia' in grupos else None,
        }

    return {'formato': 'desconocido', 'institucion': None, 'materia': None,
            'fecha': None, 'semana': semana_carpeta,
            'semana_carpeta': semana_carpeta, 'dia': None}


def revisar(meta):
    """Discrepancias detectables sin ninguna referencia externa."""
    if not VALIDAR:
        return []

    problemas = []

    # d# contra el día de la semana de la fecha: intrínseco, no necesita ancla.
    if meta['dia'] is not None and meta['fecha'] is not None:
        dia_real = meta['fecha'].weekday() + 1
        if meta['dia'] != dia_real:
            problemas.append(
                f"dice d{meta['dia']} pero {meta['fecha']} es "
                f"{DIAS_SEMANA[dia_real - 1]} (d{dia_real})"
            )

    # w# contra la carpeta que lo contiene.
    if meta['formato'] == 'nuevo' and meta['semana'] != meta['semana_carpeta']:
        problemas.append(
            f"está en week_{meta['semana_carpeta']} pero su nombre dice w{meta['semana']}"
        )

    # Materia nueva: probable error de tipeo.
    if meta['materia'] and meta['materia'] not in materias_conocidas:
        problemas.append(
            f"la materia '{meta['materia']}' no tiene carpeta en text/; "
            f"se va a crear una nueva"
        )

    return problemas


trabajo, ya_hechos, avisos = [], [], []
por_carpeta = []

for carpeta in seleccionadas:
    numero = numero_de_semana(carpeta)
    archivos = sorted(p for p in carpeta.iterdir()
                      if p.is_file() and p.suffix.lower() in EXTENSIONES_AUDIO)
    pendientes_aqui = 0

    for archivo in archivos:
        meta = interpretar(archivo.stem, numero)
        problemas = revisar(meta)

        if meta['formato'] == 'desconocido':
            problemas.append(f"el nombre no sigue la convención; va a {SIN_MATERIA}/")

        if problemas:
            avisos.append((carpeta.name, archivo.name, problemas))

        destino = DIR_TEXTO / (meta['materia'] or SIN_MATERIA) / f'{archivo.stem}.{FORMATO_SALIDA}'
        registro = (archivo, destino, meta, carpeta.name)

        if destino.exists():
            ya_hechos.append(registro)
        else:
            trabajo.append(registro)
            pendientes_aqui += 1

    por_carpeta.append((carpeta.name, len(archivos), pendientes_aqui))

# --------------------------------------------
# Logs
# --------------------------------------------

print(f"  {'semana':<14} {'audios':>7} {'pendientes':>11}")
print(f"  {'-' * 14} {'-' * 7} {'-' * 11}")
for nombre, n_audio, n_pend in por_carpeta:
    print(f"  {nombre:<14} {n_audio:>7} {n_pend:>11}")

print()
print(f"Total: {len(ya_hechos) + len(trabajo)} grabación(es) · "
      f"{len(ya_hechos)} ya transcritas · {len(trabajo)} pendientes")

if trabajo:
    print()
    print(f"  {'semana':<10} {'archivo':<38} {'fecha':<12} {'destino en text/'}")
    print(f"  {'-' * 10} {'-' * 38} {'-' * 12} {'-' * 20}")
    for archivo, destino, meta, semana in trabajo:
        fecha = str(meta['fecha']) if meta['fecha'] else '?'
        print(f"  {semana:<10} {archivo.name[:38]:<38} {fecha:<12} {destino.parent.name}")

if avisos:
    print()
    print(f"  AVISOS ({len(avisos)} archivo(s)):")
    for semana, nombre, problemas in avisos:
        print(f"    {semana}/{nombre}")
        for problema in problemas:
            print(f"      - {problema}")
    print()
    print("  Son advertencias: los archivos se transcriben igual. Revisá sobre todo las")
    print("  materias sin carpeta, que suelen ser errores de tipeo.")

print()
if trabajo:
    megabytes = sum(a.stat().st_size for a, _, _, _ in trabajo) / 1e6
    destinos = sorted({d.parent.name for _, d, _, _ in trabajo})
    print(f"A transcribir: {len(trabajo)} archivo(s), {megabytes:.0f} MB.")
    print(f"Materias involucradas: {', '.join(destinos)}")
    print("Continuá con la sección 5.")
else:
    print("No hay nada pendiente en las semanas seleccionadas.")

### Qué deberías ver

Una tabla con cada grabación, su fecha y la carpeta de `text/` a la que va a ir, más el
estado.

**El aviso que más conviene mirar es el de materia sin carpeta.** Las otras dos
verificaciones señalan un nombre mal escrito, que es molesto pero visible; una materia mal
escrita crea una carpeta nueva que parece legítima y deja esa clase separada del resto de
su materia sin que nada lo delate después.

En la primera corrida van a aparecer avisos para todas las materias, porque `text/`
todavía está vacía. Es correcto: está informando qué carpetas va a crear.

## 5. Transcripción

La celda larga, la que queda corriendo. Procesa los archivos de a uno y escribe cada
resultado en la carpeta de su materia apenas termina.

### Resistencia a la caída de la sesión

Colab no garantiza la continuidad de la sesión, de modo que la celda está escrita
asumiendo que puede morir en cualquier momento. Tres mecanismos acotan la pérdida.

* **Cada transcripción se escribe apenas termina su archivo**, no al final de todo. Una
  caída en la quinta clase conserva las cuatro anteriores.
* **Se guarda un avance parcial cada cinco minutos de audio**, con el sufijo `.parcial`.
  Una caída a mitad de una clase de dos horas conserva lo transcrito hasta el último
  guardado, en lugar de perder las dos horas.
* **La sección 4 omite lo ya hecho.** Volver a ejecutar después de una caída retoma donde
  quedó, sin repetir trabajo.

### Decisiones de implementación

* **El modelo se construye una sola vez, fuera del bucle.** Cargarlo cuesta entre diez y
  treinta segundos; hacerlo por archivo multiplicaría esa espera por la cantidad de
  clases sin ganar nada.
* **Un archivo que falla no detiene la corrida.** Se registra el error y se sigue con el
  siguiente, que es lo correcto cuando el proceso queda desatendido: un `.m4a` truncado
  no debería costar la noche entera de trabajo.
* **El encabezado hereda los metadatos del nombre.** Materia, fecha, semana y día quedan
  escritos dentro del archivo de texto, de modo que una transcripción suelta sigue siendo
  identificable fuera de su carpeta. Es también lo que permite ordenar cronológicamente
  una carpeta de materia sin depender del nombre del archivo.
* **La carpeta de la materia se crea al escribir**, no antes. Si una corrida falla entera,
  no quedan carpetas vacías sugiriendo que hay material donde no lo hay.
* **El archivo parcial se borra al completar.** Su presencia indica una transcripción
  interrumpida, de modo que conservarlo al terminar sería una señal falsa.

In [ ]:
# ============================================
# 5. Transcripción
# ============================================

import time
import traceback
from datetime import datetime

# --------------------------------------------
# Configuración
# --------------------------------------------

MODELO  = 'large-v3'   # large-v3 | large-v3-turbo | medium
IDIOMA  = 'es'         # None para detección automática

BEAM_SIZE   = 5        # 1 acelera a cambio de precisión
VAD_FILTER  = True     # descarta los silencios largos

# Cada cuántos segundos de audio se guarda el avance parcial
INTERVALO_GUARDADO = 300

# --------------------------------------------
# Procesamiento
# --------------------------------------------

def hhmmss(segundos):
    segundos = int(max(segundos, 0))
    return f'{segundos // 3600:02d}:{(segundos % 3600) // 60:02d}:{segundos % 60:02d}'


def encabezado(audio, meta, info, incompleto):
    """Frontmatter con los metadatos que se extrajeron del nombre del archivo."""
    campos = [
        '---',
        f'source: {audio.name}',
        f'institucion: {meta["institucion"] or "?"}',
        f'materia: {meta["materia"] or "?"}',
        f'semana: {meta["semana"]}',
        f'dia_semana: {meta["dia"] if meta["dia"] is not None else "?"}',
        f'fecha: {meta["fecha"] or "?"}',
        f'model: {MODELO}',
        f'language: {info.language}',
        f'duration: {hhmmss(info.duration)}',
        f'transcribed_at: {datetime.now().isoformat(timespec="seconds")}',
        'tool: oido v0.4 (colab)',
    ]
    if incompleto:
        campos.append('status: INCOMPLETO')
    return campos + ['---', '', '']


def volcar(destino, audio, meta, info, segmentos, incompleto=False):
    """Escribe la transcripción en el formato configurado, creando su carpeta."""
    if FORMATO_SALIDA in ('md', 'txt'):
        cuerpo = '\n\n'.join(s.text.strip() for s in segmentos)
        texto = '\n'.join(encabezado(audio, meta, info, incompleto)) + cuerpo + '\n'
    elif FORMATO_SALIDA == 'srt':
        bloques = []
        for n, s in enumerate(segmentos, 1):
            inicio = f'{hhmmss(s.start)},{int(s.start % 1 * 1000):03d}'
            fin    = f'{hhmmss(s.end)},{int(s.end % 1 * 1000):03d}'
            bloques.append(f'{n}\n{inicio} --> {fin}\n{s.text.strip()}\n')
        texto = '\n'.join(bloques)
    else:  # vtt
        bloques = []
        for s in segmentos:
            inicio = f'{hhmmss(s.start)}.{int(s.start % 1 * 1000):03d}'
            fin    = f'{hhmmss(s.end)}.{int(s.end % 1 * 1000):03d}'
            bloques.append(f'{inicio} --> {fin}\n{s.text.strip()}\n')
        texto = 'WEBVTT\n\n' + '\n'.join(bloques)

    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(texto, encoding='utf-8')


print(f"Cargando {MODELO}. La primera vez descarga los pesos y puede tardar.")
reloj = time.monotonic()
modelo = WhisperModel(MODELO, device='cuda', compute_type='float16')
print(f"Modelo listo en {time.monotonic() - reloj:.0f}s")
print()

completados, fallados = [], []
arranque = time.monotonic()

semana_en_curso = None

for indice, (audio, destino, meta, semana) in enumerate(trabajo, 1):
    # Encabezado al cambiar de semana, para seguir una corrida larga.
    if semana != semana_en_curso:
        semana_en_curso = semana
        print(f"--- {semana} " + "-" * (58 - len(semana)), flush=True)
    print(f"[{indice}/{len(trabajo)}] {destino.parent.name} · {audio.name}", flush=True)
    parcial = destino.with_suffix(f'.parcial.{FORMATO_SALIDA}')

    try:
        inicio = time.monotonic()
        segmentos_iter, info = modelo.transcribe(
            str(audio),
            language=IDIOMA,
            beam_size=BEAM_SIZE,
            vad_filter=VAD_FILTER,
        )
        print(f"    duración {hhmmss(info.duration)} · idioma {info.language}", flush=True)

        acumulados, ultimo_guardado = [], 0.0

        for segmento in segmentos_iter:
            acumulados.append(segmento)

            if segmento.end - ultimo_guardado >= INTERVALO_GUARDADO:
                ultimo_guardado = segmento.end
                volcar(parcial, audio, meta, info, acumulados, incompleto=True)
                avance = 100 * segmento.end / max(info.duration, 1e-9)
                ritmo  = segmento.end / max(time.monotonic() - inicio, 1e-9)
                print(f"    {avance:5.1f}%   {ritmo:4.1f}x realtime", flush=True)

        volcar(destino, audio, meta, info, acumulados)
        parcial.unlink(missing_ok=True)

        transcurrido = time.monotonic() - inicio
        completados.append((audio.name, destino.parent.name, info.duration,
                            transcurrido, semana))
        print(f"    OK  text/{destino.parent.name}/{destino.name}  ·  "
              f"{hhmmss(info.duration)} en {hhmmss(transcurrido)}  "
              f"({info.duration / max(transcurrido, 1e-9):.1f}x)", flush=True)
        print(flush=True)

    except Exception as error:
        fallados.append((audio.name, str(error)))
        print(f"    FALLÓ: {error}", flush=True)
        traceback.print_exc()
        print(flush=True)

# --------------------------------------------
# Logs
# --------------------------------------------

total = time.monotonic() - arranque

print("=" * 66)
semanas_tocadas = sorted({s for _, _, _, _, s in completados})
print(f"Corrida terminada en {hhmmss(total)}")
if semanas_tocadas:
    print(f"  Semanas       : {', '.join(semanas_tocadas)}")
print(f"  Completados : {len(completados)}")
print(f"  Fallados    : {len(fallados)}")

for nombre, error in fallados:
    print(f"    - {nombre}: {error}")

if completados:
    audio_total = sum(d for _, _, d, _, _ in completados)
    print()
    print(f"  Audio procesado : {hhmmss(audio_total)}")
    print(f"  Velocidad media : {audio_total / max(total, 1e-9):.1f}x realtime")
    print()
    print("  Archivado en:")
    for materia in sorted({m for _, m, _, _, _ in completados}):
        n = sum(1 for _, mm, _, _, _ in completados if mm == materia)
        print(f"    text/{materia}/  ({n} archivo(s) nuevo(s))")

### Qué deberías ver

Por cada archivo: la materia a la que va, la duración detectada, el avance en porcentaje
con la velocidad, y la confirmación con la ruta final.

**La velocidad es el número a vigilar.** Sobre una T4 con `large-v3` debería rondar entre
8 y 15 veces el tiempo real. Si aparece por debajo de 2, la corrida está sobre CPU: volvé
a la sección 1 y verificá que haya GPU asignada.

Si aparece `CUDA out of memory`, bajá `MODELO` a `medium` y volvé a ejecutar. Lo ya
transcrito no se repite.

Mientras esto corre, **dejá la pestaña abierta y evitá que la computadora se suspenda**.
La pantalla puede apagarse; la máquina no. Si se suspende, el navegador se desconecta y
Colab termina la sesión.

## 6. Estado general

Recorre toda la estructura y la reporta por sus dos ejes: las semanas de entrada y las
materias de salida. Cada uno responde una pregunta distinta.

### Decisiones de implementación

* **El informe se arma leyendo Drive, no acumulando variables de las celdas anteriores.**
  Si la sesión se cayó y se retomó, las variables en memoria reflejan solo la última
  corrida; el disco refleja el estado real.
* **Se reportan los dos ejes.** La vista por semana responde "¿qué me falta procesar?";
  la vista por materia responde "¿cuánto material tengo de cada asignatura?". Ninguna de
  las dos se deduce cómodamente de la otra.
* **Los archivos `.parcial` se reportan aparte.** Su presencia indica una transcripción
  interrumpida, que conviene rehacer antes de darla por buena.
* **Las carpetas con nombre irregular se reportan, no se interpretan.** Drive crea
  duplicados llamados `week_7 (1)` cuando se sube dos veces la misma carpeta. Tratarlas
  como si fueran `week_7` haría que los audios se contaran dos veces; ignorarlas en
  silencio las dejaría invisibles. Se listan aparte para que se limpien.

In [ ]:
# ============================================
# 6. Estado general
# ============================================

# --------------------------------------------
# Procesamiento
# --------------------------------------------

# Eje de entrada: cuántos audios tiene cada semana y cuántos están transcritos.
# Se cuenta buscando el .md correspondiente en todo text/, porque la materia de cada
# audio determina en qué carpeta quedó.
transcritos = {p.stem for p in DIR_TEXTO.rglob(f'*.{FORMATO_SALIDA}')
               if '.parcial.' not in p.name}

# Las que esta corrida tocó, para marcarlas en el informe.
procesadas_en_esta_corrida = {c.name for c in seleccionadas}

semanas, semanas_irregulares = semanas_ordenadas(DIR_AUDIOS)

por_semana = []
for carpeta in semanas:
    audios = [p for p in carpeta.iterdir()
              if p.is_file() and p.suffix.lower() in EXTENSIONES_AUDIO]
    hechos = sum(1 for p in audios if p.stem in transcritos)
    por_semana.append((carpeta.name, len(audios), hechos, len(audios) - hechos))

# Eje de salida: cuántas transcripciones acumula cada materia.
por_materia, parciales = [], []
for carpeta in sorted(p for p in DIR_TEXTO.iterdir() if p.is_dir()):
    textos = [p for p in carpeta.glob(f'*.{FORMATO_SALIDA}') if '.parcial.' not in p.name]
    parciales += list(carpeta.glob(f'*.parcial.{FORMATO_SALIDA}'))
    por_materia.append((carpeta.name, len(textos)))

# --------------------------------------------
# Logs
# --------------------------------------------

print("ENTRADA — audios por semana")
print(f"  {'semana':<10} {'audios':>7} {'transcritos':>12} {'pendientes':>11}")
print(f"  {'-' * 10} {'-' * 7} {'-' * 12} {'-' * 11}")
for nombre, n_audio, n_hecho, faltan in por_semana:
    marca  = '' if faltan == 0 else '  <-'
    actual = '*' if nombre in procesadas_en_esta_corrida else ' '
    print(f" {actual}{nombre:<10} {n_audio:>7} {n_hecho:>12} {faltan:>11}{marca}")

if not por_semana:
    print("  (no hay carpetas week_N en audios/)")

if semanas_irregulares:
    print()
    print("  CARPETAS CON NOMBRE IRREGULAR (se ignoran):")
    for carpeta in semanas_irregulares:
        n = len([p for p in carpeta.iterdir()
                 if p.is_file() and p.suffix.lower() in EXTENSIONES_AUDIO])
        print(f"    {carpeta.name}  ({n} audio(s) sin procesar)")
    print()
    print("  Suelen ser duplicados que crea Drive al subir dos veces la misma carpeta.")
    print("  Mové su contenido a la week_N que corresponda y borralas.")

print()
print("SALIDA — transcripciones por materia")
print(f"  {'materia':<20} {'transcripciones':>16}")
print(f"  {'-' * 20} {'-' * 16}")
for nombre, n in por_materia:
    print(f"  {nombre:<20} {n:>16}")

if not por_materia:
    print("  (todavía no hay carpetas en text/)")

print()
print(f"  (*) semanas incluidas en esta corrida")
print(f"  Total: {sum(n for _, n in por_materia)} transcripciones de "
      f"{sum(a for _, a, _, _ in por_semana)} grabaciones")

faltantes = [n for n, _, _, f in por_semana if f > 0]
if faltantes:
    print()
    print(f"  Semanas con pendientes: {', '.join(faltantes)}")
    print("  Ajustá SEMANAS en la sección 3 y ejecutá de ahí en adelante.")

if parciales:
    print()
    print(f"  {len(parciales)} transcripción(es) INTERRUMPIDA(S):")
    for p in parciales:
        print(f"    - {p.relative_to(DIR_TEXTO)}")
    print()
    print("  Borrá esos archivos .parcial y volvé a ejecutar las secciones 4 y 5.")

### Qué deberías ver

Dos tablas. La de arriba te dice qué semanas quedan por procesar; la de abajo, cuánto
material acumulaste de cada materia.

La segunda tabla es la que vas a mirar al preparar un parcial, y es también la que te dice
qué carpeta entregarle a un asistente para que responda sobre esa asignatura. El Anexo C
desarrolla eso.

Para procesar otras semanas no hace falta reiniciar nada: ajustá `SEMANAS` en la sección 3
y ejecutá de ahí en adelante. El modelo ya está cargado en memoria, así que la siguiente
corrida arranca sin la espera inicial.

## Anexo A. Automatización en el teléfono

El notebook asume que las grabaciones llegan a `audios/week_N/` ya nombradas. Esta es la
parte que ocurre en el Poco X7 Pro.

### Qué hay que indicar y qué se deriva

El formato tiene seis campos. Vale mirar de dónde sale cada uno antes de programar nada:

| Campo | De dónde sale |
|---|---|
| `ucab` | Constante. No cambia nunca. |
| `w2` | **Variable global que se actualiza una vez por semana.** |
| `d4` | Se deriva: día de la semana de la fecha del archivo. |
| `apispy` | **Lo único que se elige por grabación.** |
| `24-09-2026` | Es la fecha del archivo, que el teléfono ya guarda. |

De modo que el atajo no necesita un formulario de cinco campos. Necesita **una lista de
materias y un toque**, más una variable que se cambia los lunes.

La semana es una variable y no un cálculo justamente por lo que motivó esta versión: no
hace falta declarar cuándo empezó el semestre ni mantener esa referencia. Se dice en qué
semana se está, una vez, y se acabó.

### Qué aplicación usar

Android no trae un sistema de atajos equivalente al de iOS, así que hace falta una
aplicación de automatización. Para este flujo conviene **Tasker**, porque tiene juntas las
tres piezas necesarias: diálogos de selección, operaciones de archivo y un intérprete de
JavaScript. MacroDroid es más fácil de usar pero su manejo de archivos es más limitado.

Tasker es de pago, unos pocos dólares, con período de prueba.

### Variables globales

Se definen una vez en `Vars`:

```
%INSTITUCION   ucab
%SEMANA        2                 ← se actualiza cada lunes
%ORIGEN        /storage/emulated/0/MIUI/sound_recorder
%DESTINO       /storage/emulated/0/oido_pendientes
%MATERIAS      apispy,bd,ciberoff,ciberofflab,softwaredev
```

La ruta de `%ORIGEN` depende del grabador. Para encontrarla, grabá una nota de prueba y
buscá el archivo con el explorador de archivos del teléfono.

**Que `%MATERIAS` coincida exactamente con los nombres de las carpetas de `text/`** es lo
que evita que una variante cree una carpeta duplicada. Si la carpeta es `ciberofflab`, la
lista dice `ciberofflab`, no `ciberoff-lab`.

### Tarea — Nombrar y archivar

1. **`List Dialog`** → `Items: %MATERIAS` → guarda en `%MATERIA`.
   Este es el único momento en que el atajo pide algo.

2. **Obtener el archivo.** Acá está la diferencia entre los dos modos de uso:
   * **Después de cada clase**: `File Select` sobre `%ORIGEN`, o el más reciente.
   * **Al final de la semana**: `For` sobre `%ORIGEN`, repitiendo los pasos 1 a 5 por cada
     archivo, de modo que se indique la materia de cada uno.

3. **`Test File`** → `Type: Modified` → guarda en `%MODIFICADO`.
   De acá sale la fecha de la clase sin preguntarla.

4. **`JavaScriptlet`** con el código de abajo. Calcula `%DIA` y `%FECHA_NOMBRE`.

5. **`File Move`** → de la ruta del archivo a:
   ```
   %DESTINO/%INSTITUCION-w%SEMANA-d%DIA-%MATERIA-%FECHA_NOMBRE.m4a
   ```

### El JavaScriptlet

Más corto que en versiones anteriores, porque ya no calcula la semana:

```javascript
// Entrada: %MODIFICADO (milisegundos, de Test File)
// Salida:  %DIA (1-7, lunes a domingo) y %FECHA_NOMBRE (DD-MM-YYYY)

var mod = new Date(parseInt(local('MODIFICADO')));

// Se normaliza a mediodía UTC para que ningún cambio de huso corra la fecha un día.
var clase = new Date(Date.UTC(mod.getFullYear(), mod.getMonth(), mod.getDate(), 12));

var d = clase.getUTCDay();                    // 0 = domingo
setLocal('DIA', d === 0 ? 7 : d);             // 1 = lunes … 7 = domingo

setLocal('FECHA_NOMBRE',
    ('0' + clase.getUTCDate()).slice(-2) + '-' +
    ('0' + (clase.getUTCMonth() + 1)).slice(-2) + '-' +
    clase.getUTCFullYear());
```

La normalización a mediodía UTC no es un detalle ocioso: una clase grabada a las 23:00 y
procesada con la fecha en hora local puede caer en el día siguiente al convertir de huso,
y el `d#` saldría corrido. Fijar la hora a mediodía deja un margen de doce horas para cada
lado.

### Subida a Drive

Para esto no hace falta otra tarea. **Autosync for Google Drive** (MetaCtrl) vigila la
carpeta y sube sola.

1. Instalá Autosync y autorizá tu cuenta.
2. Creá un par de carpetas:
   * **Local**: `oido_pendientes` (el `%DESTINO` de la tarea anterior).
   * **Remota**: `google_colab/transcriptioner/audios/week_N`.
   * **Método**: *Upload only*. Con sincronización bidireccional, cualquier cosa que se
     borre de Drive se borraría también del teléfono.
3. Activá **solo con WiFi**. Una clase de dos horas pesa entre 60 y 120 MB.

La carpeta remota lleva el número de semana, así que hay que cambiarla cada lunes, junto
con `%SEMANA` en Tasker. Son dos ediciones semanales en el mismo momento.

### Si Tasker resulta demasiado

Hay un camino sin automatización que funciona: grabar, y al final de la semana renombrar
los archivos a mano desde el explorador del teléfono antes de subirlos. La convención está
pensada para escribirse a mano sin ambigüedad, y la sección 4 del notebook avisa si algo
quedó mal.

## Anexo B. Rutina semanal

| Cuándo | Qué |
|---|---|
| Cada lunes | Actualizar `%SEMANA` en Tasker y la carpeta remota en Autosync. |
| Después de cada clase | Ejecutar la tarea de Tasker y elegir la materia. |
| Fin de semana | Verificar que Autosync subió todo a `audios/week_N/`. |
| | Abrir el notebook y asignar GPU. |
| | Ejecutar las secciones 1 y 2 (entorno y motor). |
| | Ajustar `SEMANAS` en la sección 3 (o dejarlo en `[]` para todas). |
| | Ejecutar las secciones 3 a 6. |

Con `SEMANAS = []` la corrida recorre todas las semanas pendientes de una sola vez, que es
lo cómodo cuando hay atraso acumulado.

### Parámetros que se tocan

| Parámetro | Sección | Cuándo cambiarlo |
|---|---|---|
| `SEMANAS` | 3 | En cada corrida |
| `MODELO` | 5 | Si falta memoria o sobra tiempo |
| `FORMATO_SALIDA` | 4 | Si querés subtítulos en vez de texto |
| `VALIDAR` | 4 | Para silenciar los avisos de nombre |

### Si algo falla

| Síntoma | Causa probable | Qué hacer |
|---|---|---|
| `Unable to load libcudnn...` | Enlazador inconsistente | Reiniciar sesión, ejecutar desde la sección 2 |
| `CUDA out of memory` | Modelo grande para la GPU | `MODELO = 'medium'` en la sección 5 |
| Menos de 2x realtime | Corriendo sobre CPU | Verificar la sección 1 |
| Sesión cortada | Inactividad o suspensión | Reejecutar de la sección 3; omite lo ya hecho |
| Semana pedida que no existe | Sin subir, o número errado | Ver el listado que imprime la sección 3 |
| Carpeta de materia duplicada | Variante en el nombre | Unificar `%MATERIAS` con las carpetas de `text/` |
| Aviso de `d#` incoherente | Fecha del teléfono o renombrado manual | Corregir el nombre antes de transcribir |

## Anexo C. Una materia, un proyecto de Claude

La estructura `text/<materia>/` no es solo un criterio de archivo: es la unidad que se le
puede entregar a un asistente para que responda sobre esa asignatura.

### Proyectos

En Claude.ai, un **Proyecto** es un espacio de trabajo con su propio historial de
conversaciones y su propia base de conocimiento. Dos características lo hacen encajar con
esta estructura:

* **Instrucciones propias.** Cada proyecto admite instrucciones que se aplican a todas las
  conversaciones dentro de él. Es el "prompt de sistema por carpeta".
* **Base de conocimiento.** Se suben documentos al proyecto y quedan disponibles como
  contexto en cada conversación, sin volver a adjuntarlos.

La correspondencia con la estructura de Drive es directa: **una carpeta de `text/` es un
proyecto**.

```
text/apispy/        →  Proyecto "APIs en Python"
text/bd/            →  Proyecto "Bases de Datos"
text/ciberoff/      →  Proyecto "Ciberseguridad Ofensiva"
```

Las cuentas gratuitas admiten hasta cinco proyectos; para más materias hace falta un plan
de pago.

### Cómo armarlo

1. En Claude.ai, crear un proyecto por materia.
2. Escribir las instrucciones del proyecto. Lo que más rinde es decir qué es el material y
   cómo se espera que lo use:

   > Este proyecto contiene transcripciones automáticas de las clases de Desarrollo de
   > APIs en Python de la UCAB. Cada archivo corresponde a una clase y empieza con un
   > encabezado que indica la materia, la semana, el día y la fecha.
   >
   > Las transcripciones provienen de un reconocimiento automático de voz: los términos
   > técnicos pueden estar mal transcritos y la puntuación es aproximada. Cuando algo no
   > se entienda, decilo en lugar de completarlo por tu cuenta.
   >
   > Al responder, citá la clase de la que sale cada cosa usando su semana y fecha.
   > Si algo no está en las transcripciones, decilo explícitamente en vez de responder
   > con conocimiento general.

3. Subir los `.md` de esa materia a la base de conocimiento del proyecto.

El tercer párrafo de esas instrucciones es el que más importa. Sin él, la respuesta mezcla
lo que dijo el profesor con conocimiento general del tema, y deja de servir para estudiar
para un parcial: lo que se evalúa es lo que se dio en clase.

### Por qué el encabezado del archivo importa acá

Cada transcripción empieza con su frontmatter:

```yaml
---
materia: apispy
semana: 2
dia_semana: 4
fecha: 2026-09-24
---
```

Eso es lo que permite pedir "resumí lo de la semana 3" o "en qué clase se vio
autenticación" y obtener una respuesta que señale la clase concreta. Sin esos campos, los
archivos serían texto plano indistinguible y habría que abrirlos uno por uno.

Es también la razón por la que conviene atender los avisos de la sección 4: un `d#` mal
escrito termina dentro del encabezado, y de ahí a una respuesta que ubica mal una clase.

### Qué subir y cada cuánto

Las transcripciones de una materia son de pocas decenas de kilobytes cada una, de modo que
un semestre entero cabe sin problema. Lo razonable es subir el lote de la semana después
de cada corrida del notebook.

Para una consulta puntual sobre una sola clase no hace falta proyecto: alcanza con
adjuntar ese `.md` a una conversación. El proyecto rinde cuando la pregunta cruza varias
clases, que es el caso al estudiar.

Fuente: [Proyectos en Claude.ai](https://support.claude.com/en/articles/9517075-what-are-projects)